### Exhaustive 3: Tools (Function Calling) Deep Dive

This notebook breaks down `3-tools.py`, where the model gets live weather data by asking **your code** to run a function.

**The key point:** The model never runs code; it can only write text. "Calling a tool" means the model writes a small request like "please run `get_weather` with these two numbers", and your Python code does the actual running. That's why one tool use needs **two API calls with your code in between**.

##### The whole flow:
```
1. Call 1: client.chat.completions.create(messages, tools)
   The model reads the question + the tool's name/description/parameters
   and replies with a REQUEST, not an answer:
   name='get_weather', arguments='{"latitude":59.9139,"longitude":10.7522}'
            │
            ▼
2. Your code (no model involved):
   json.loads(arguments) -> call_function -> get_weather(**args)
   -> real HTTP request to api.open-meteo.com -> {"temperature_2m": 14.5, ...}
   Append the model's request and the result to `messages`.
            │
            ▼
3. Call 2: client.chat.completions.parse(messages, tools, response_format)
   The model reads the whole history, including the result,
   and writes the answer -> WeatherResponse(temperature=14.5, response='...')
```

- **Section 1**: What the model reads: the tool `description`, not the docstring.
- **Section 2**: Reading the response of call 1.
- **Section 3**: Call 1 did not fetch the weather, so why the rest?
- **Section 4**: `call_function`, `**args`, and the loop line by line.
- **Section 5**: The four messages the second call receives.
- **Section 6**: Call 2, and where `Field()` descriptions go.
- **Section 7**: Checking the final answer against what the model was given.


In [1]:
import json
import os
from pprint import pprint
import requests
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
load_dotenv()
client = OpenAI()

In [3]:
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto&wind_speed_unit=ms"
    )
    data = response.json()
    # send the units and timezone too, so the model doesn't have to guess them
    return {
        "current": data["current"],
        "units": data["current_units"],
        "timezone": data["timezone"],
    }

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]


#### 1. What the model reads: the tool `description`, not the docstring

**Key point:** The model reads only the JSON request body that the SDK sends, and the Python function `get_weather` is never part of it. Its docstring never leaves your machine. Everything the model knows about the tool comes from the `tools` list, so the text that plays the docstring's role is `"description"`.

Look at the call in the next cell: `create(model=..., messages=..., tools=...)`. `get_weather` isn't passed anywhere. The only link between the model and your function is the **name string** `"get_weather"`, which you wrote in both places.

##### The exact request body of call 1
This is what the SDK sends for the first call. I captured it with a fake HTTP transport, so nothing went to OpenAI. The docstring text ("This is a publically available API...") appears nowhere in it.

<div style="font-size: 0.85em">

```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"}
  ],
  "model": "gpt-5-nano",
  "tools": [{
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current temperature for provided coordinates in celsius.",
      "parameters": {
        "type": "object",
        "properties": {"latitude": {"type": "number"}, "longitude": {"type": "number"}},
        "required": ["latitude", "longitude"],
        "additionalProperties": false
      },
      "strict": true
    }
  }]
}
```

</div>

##### What the model uses each part of the tool for
- **`name`**: The label the model writes back when it wants this tool. Your `call_function` matches on this string.
- **`description`**: The only prose about the tool. The model uses it to decide *whether* this tool fits the question; with ten tools, this is how it picks one. Here the docstring and the description say different things, and only the description affects the model.
- **`parameters`**: A JSON Schema that tells the model which argument names and types to write. `"strict": true` makes the API constrain generation so the arguments always match this schema. It's the same constrained decoding you saw with `response_format` in Exhaustive 2.

##### How this relates to `Field(description=...)`
`Field` descriptions reach the model because the SDK converts your Pydantic class into a JSON Schema and puts it into the request body (you'll see it in Section 6). A plain function gets no such conversion; nothing reads its signature or docstring.

The tool equivalent of `Field(description=...)` is a `"description"` key inside each property. The course code has none, but you could write:
```python
"properties": {
    "latitude": {"type": "number", "description": "Latitude in decimal degrees, e.g. 59.91 for Oslo."},
    "longitude": {"type": "number", "description": "Longitude in decimal degrees, e.g. 10.75 for Oslo."},
},
```

Two related facts, so the rule doesn't get over-simplified:
- A docstring on a **Pydantic class** *is* sent, because Pydantic copies it into the schema as the top-level `"description"`. If you add `"""Final answer about the weather."""` under `class WeatherResponse(BaseModel):`, the schema the SDK sends starts with `{'description': 'Final answer about the weather.', 'properties': {...`.
- Some agent frameworks (e.g. the OpenAI Agents SDK's `@function_tool`) build the tool definition *from* a function's signature and docstring. The docstring reaches the model there only because the framework copies it into `"description"`. The plain `OpenAI()` client used here doesn't do that.

**The rule behind both:** The model only sees what is in the request body, so something has to copy text into it. Pydantic does that for a class (it writes the class into the schema), and `@function_tool` does it for a function. With a plain function and the plain client, nothing does: `get_weather` itself is never passed to the API, only the `tools` dict you wrote by hand.

In [5]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [6]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EQHT6WVrDbQhCrNDqSg2DYcgLRhMC",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_RxBcijsSCcyqeEV4DECORc9Z",
            "function": {
              "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1789932384,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 225,
    "prompt_tokens": 150,
    "total_tokens": 375,
    "complet

In [7]:
pprint(completion.model_dump(), sort_dicts=False)   #"pretty print"

{'id': 'chatcmpl-EQHT6WVrDbQhCrNDqSg2DYcgLRhMC',
 'choices': [{'finish_reason': 'tool_calls',
              'index': 0,
              'logprobs': None,
              'message': {'content': None,
                          'refusal': None,
                          'role': 'assistant',
                          'annotations': [],
                          'audio': None,
                          'function_call': None,
                          'tool_calls': [{'id': 'call_RxBcijsSCcyqeEV4DECORc9Z',
                                          'function': {'arguments': '{"latitude":59.9139,"longitude":10.7522}',
                                                       'name': 'get_weather'},
                                          'type': 'function'}]}}],
 'created': 1789932384,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'metadata': None,
 'moderation': None,
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 225,
           'prompt

#### 2. Reading the response: the model *asked* for a function, and nothing ran

**Key point:** The model's entire reply to call 1 is the request "run `get_weather` with `{"latitude":59.9139,"longitude":10.7522}`". There is no answer text (`content=None`), and `finish_reason='tool_calls'` says the model stopped because it wants a tool run. Everything else in the output is bookkeeping.

The two outputs above are **the same data in two forms**. Both come from Pydantic methods, which exist here because `ChatCompletion` is a Pydantic `BaseModel`:
- `completion.model_dump_json(indent=2)` returns a **string** of JSON (`null`, double quotes). `indent=2` puts one field per line.
- `completion.model_dump()` returns a Python **dict** (`None`, single quotes). `pprint` ("pretty print", from Python's standard library, imported at the top) prints it one key per line, and `sort_dicts=False` keeps the original key order instead of sorting the keys alphabetically.

A plain `print(completion)` would print the object in its own form, `ChatCompletion(id='chatcmpl-...', choices=[Choice(...)], ...)`, all on one line. The class names from that form are the ones in brackets in the tree below.

##### The tree (only the parts that matter; the rest are `None` or empty here)

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
completion  <span style="opacity: 0.55">(ChatCompletion)</span>
├─ id        'chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB'
├─ created   1789801298                 <span style="opacity: 0.7">← Unix time</span>
├─ model     'gpt-5-nano-2025-08-07'    <span style="opacity: 0.7">← exact model version</span>
├─ choices   <span style="opacity: 0.55">(list, 1 item)</span>
│  └─ [0]  <span style="opacity: 0.55">(Choice)</span>
│     ├─ finish_reason  'tool_calls'    <span style="opacity: 0.7">← it wants a tool run</span>
│     └─ message  <span style="opacity: 0.55">(ChatCompletionMessage)</span>
│        ├─ role        'assistant'
│        ├─ content     None            <span style="opacity: 0.7">← no answer text</span>
│        └─ tool_calls  <span style="opacity: 0.55">(list, 1 item)</span>
│           └─ [0]  <span style="opacity: 0.55">(ChatCompletionMessageFunctionToolCall)</span>
│              ├─ id        'call_jVJ67G2TzgPRcCCApHW4ya7r'
│              ├─ type      'function'
│              └─ function  <span style="opacity: 0.55">(Function)</span>
│                 ├─ name       'get_weather'
│                 └─ arguments  '{"latitude":59.9139,"longitude":10.7522}'
└─ usage     <span style="opacity: 0.55">(CompletionUsage)</span>
   ├─ prompt_tokens      150            <span style="opacity: 0.7">← tokens read</span>
   ├─ completion_tokens  289            <span style="opacity: 0.7">← tokens written (256 reasoning)</span>
   └─ total_tokens       439
</pre>

The same path reaches the arguments string `'{"latitude":59.9139,"longitude":10.7522}'` in both forms:
```python
# object: dots
completion.choices[0].message.tool_calls[0].function.arguments

# dict: keys
d = completion.model_dump()
d["choices"][0]["message"]["tool_calls"][0]["function"]["arguments"]
```

Things worth noticing:
- **`finish_reason`** is `'tool_calls'` here. For a normal text answer it's `'stop'`.
- **`arguments` is in quotes:** It's a **string** that contains JSON, not a dict. Section 4 turns it into a dict with `json.loads`.
- **The tool call's `id`** (`'call_jVJ67G2TzgPRcCCApHW4ya7r'`) is how the result you send back gets paired with this request (Section 4).
- **Where the coordinates came from:** Nothing in the request contained Oslo's latitude and longitude. The model supplied them from its training knowledge. That is the part of the job the model does: choosing the tool and filling in its arguments.
- **Reasoning tokens:** `gpt-5-nano` is a reasoning model, so it thinks privately before writing. Of the 289 tokens it wrote, 256 were that hidden reasoning (`usage.completion_tokens_details.reasoning_tokens`), and only the remaining 33 were the tool call itself. You pay for all 289.
- **The `None`/empty fields** belong to features not used here: `logprobs`, `audio`, `refusal` (filled if the model declines), `annotations` (e.g. web-search citations), and `function_call` (the older, deprecated form of `tool_calls`).

##### Walking those paths in code
The cell below reads exactly those fields off the object, with the dotted form. It parks `completion.choices[0].message` in a variable called `reply`, deliberately **not** `message`: that name sits one letter away from `messages` and invites the misreading that your `messages` list has already received the model's answer. It hasn't, and Section 4 works through why. `tool_calls` is a list, so the loop is what handles a reply that asks for two tools at once. `repr()` prints the quotes, which is what makes `arguments` visibly a string rather than a dict, and `type(...).__name__` says so outright.

In [8]:
reply = completion.choices[0].message   # named `reply`, NOT `message`, to keep it distinct from your `messages` list

print("finish_reason:", completion.choices[0].finish_reason)
print("content:      ", reply.content)
for tool_call in reply.tool_calls:
    print("tool call:")
    print("   id:        ", tool_call.id)
    print("   name:      ", tool_call.function.name)
    print("   arguments: ", repr(tool_call.function.arguments), f"({type(tool_call.function.arguments).__name__})")

finish_reason: tool_calls
content:       None
tool call:
   id:         call_RxBcijsSCcyqeEV4DECORc9Z
   name:       get_weather
   arguments:  '{"latitude":59.9139,"longitude":10.7522}' (str)


In [9]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"}]

#### 3. Call 1 did not fetch the weather, so why the rest?

**Key point:** `tools` + `messages` + `create()` only get the model to *choose* a tool and *write its arguments*. They can't fetch anything, because the model runs on OpenAI's servers with no access to your function and can't execute Python. The weather is fetched by your code in the next cell. Then a second API call is needed so the model can read the result and write the answer.

Three pieces of evidence from the outputs above:
1. `content=None`: the model wrote no answer, only a tool request.
2. `messages` (the cell just above) still holds only the 2 messages you wrote. `create()` doesn't add anything to your list, and the API keeps no memory between calls. Whatever the model should see next time, *you* have to append.
3. Open-Meteo hasn't been contacted yet. The only code that sends a request to `api.open-meteo.com` is the body of `get_weather`. It runs on your machine, and nothing has called it so far.

So the roles are:
- **Completion 1** decides *which* function to run and *with which arguments*. It does not retrieve the information.
- **Your code** (the next cell) retrieves the information.
- **Completion 2** reads that information and writes the answer, here in the `WeatherResponse` shape.

#### 4. Running the tool ourselves: `call_function`, `**args`, and the loop

##### `call_function` is a dispatcher
**What it is:** A function that takes the tool name the model wrote and runs the matching Python function.

**Why it's needed:** The model only gives you the *string* `'get_weather'`, and you can't call a string. `call_function` maps the name to the real function. With one tool it's a single `if`. With several tools you add one branch per tool (or use a dict like `{"get_weather": get_weather}`).

##### `**args` unpacks a dict into keyword arguments
**What it does:** `**` inside a function call takes each `key: value` pair of a dict and passes it as `key=value`.

This checks that `json.loads` turns the model's arguments string into a dict:
```python
arguments = '{"latitude":59.9139,"longitude":10.7522}'
print(type(arguments))
args = json.loads(arguments)
print(args)
print(type(args))
```
```
<class 'str'>
{'latitude': 59.9139, 'longitude': 10.7522}
<class 'dict'>
```

This checks that `get_weather(**args)` is the same call as writing the keywords out by hand. It uses a stub `get_weather` that returns its inputs instead of calling the API:
```python
def get_weather(latitude, longitude):
    return f"called with latitude={latitude}, longitude={longitude}"

print(get_weather(**args))
print(get_weather(latitude=59.9139, longitude=10.7522))
```
```
called with latitude=59.9139, longitude=10.7522
called with latitude=59.9139, longitude=10.7522
```

This works only because the property names in the tool's `parameters` (`latitude`, `longitude`) exactly match the Python parameter names. This checks what happens when they don't:
```python
get_weather(**{"lat": 59.9139, "lon": 10.7522})
```
```
TypeError: get_weather() got an unexpected keyword argument 'lat'
```

##### Why the model's reply has to be appended at all
**Key point:** `messages` is a list *you* own. `create()` reads it and never writes to it, and the API stores nothing between calls. After call 1, `messages` still holds only the two entries you typed. The model's reply arrived in a **different variable**, `completion`. If you don't copy it across yourself, call 2 never sees it.

The two variables are easy to mix up:

- `messages` is **your input**. A plain Python `list` that you build and append to. It is the model's entire memory.
- `completion` is **the API's output**. A `ChatCompletion` object handed back by `create()`. The assistant's reply sits at `completion.choices[0].message`.

The bare `messages` cell above (run after call 1, before any append) prints:
```
[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"}]
```

Two entries, both written by you. The tool call with `get_weather` and id `call_jVJ6...` is **not** in this list — it lives inside `completion`, which the cells above printed separately. So the line

```python
messages.append(completion.choices[0].message)
```

is what moves the reply out of the output object and into your input list. Nothing does it for you. The remaining question is only *how many times* that line should run, which is what the rest of this section works out.

##### The loop, line by line
`completion.choices[0].message.tool_calls` is a **list**, because the model can ask for several calls in one reply (e.g. "weather in Oslo and Paris?" gives two `get_weather` calls). The loop handles each one.

Before the loop, once:
- `messages.append(completion.choices[0].message)` copies the model's tool request into `messages`. Note *what* gets copied: the whole assistant message from call 1, with every entry of `tool_calls` still nested inside it.

**Why this line can't be skipped.** The API enforces a pairing rule: a `"role": "tool"` message is legal only if an **earlier** assistant message *in the same list you send* carries a `tool_calls` entry with the same id. A `tool_call_id` is a pointer, and the thing it points at has to travel with it.

Skip the append, and call 2 receives three entries:

```
1  system
2  user
3  tool     answers call_jVJ6...  <- an id that appears nowhere above
```

Entry 3 claims to answer a request this list never contains, so the API rejects the whole call. Run live, that is a 400:

<div style="font-size: 0.85em">

```
BadRequestError: Error code: 400 - {'error': {'message':
"Invalid parameter: messages with role 'tool' must be a response
to a preceeding message with 'tool_calls'.",
'type': 'invalid_request_error',
'param': 'messages.[2].role', 'code': None}}
```

</div>

(`preceeding` is the API's own spelling.) As with the Version A error later in this section, `'param'` is 0-indexed, so `messages.[2]` is entry 3 — the orphaned tool message.

That gives the one line three possible placements, and only the last is correct:
- **Never appended:** Call 2 carries a tool result answering no request. 400, shown above.
- **Appended inside the loop:** With two or more tool calls the request is duplicated, which splits a request away from its answers. 400, worked through below.
- **Appended once, above the loop:** One request entry, then one tool entry per id. Accepted.

Inside the loop, once per tool call:
- `name = tool_call.function.name` gives `'get_weather'`, which your code uses to pick the function.
- `args = json.loads(tool_call.function.arguments)` turns the string into a dict (shown above) so Python can use it.
- `result = call_function(name, args)` is **where `get_weather` actually runs** and the real HTTP request goes to Open-Meteo. In the saved run (before the Section 7 fix) it returned `{'time': '2026-09-19T07:15', 'interval': 900, 'temperature_2m': 14.5, 'wind_speed_10m': 14.4}`. Now that dict comes back under `"current"`, next to `"units"` and `"timezone"`.
- `messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})` hands the result back:
  - `"role": "tool"` marks the message as a tool result, not something the user said.
  - `"tool_call_id"` pairs the result with the request's id (`call_jVJ6...`). With several calls, this tells the model which result answers which request.
  - `json.dumps(result)` turns the dict into a string, because a message's `content` must be text.

The two conversions go in opposite directions: `json.loads` converts **string → dict** for your code, and `json.dumps` converts **dict → string** for the model.

##### Why the first append is above the loop (changed from the course code)
**Key point:** The model's request is **one message that can hold several tool calls**, not one message per call. The course code appends that one message once per pass of the loop, so with two tool calls it lands in the history twice, and call 2 is rejected.

**The run.** I asked *"What's the weather like in Oslo and in Paris today?"* and ran both versions against the real API. Call 1 came back as **one** assistant message carrying **two** tool calls:

```
completion.choices[0].message      <- ONE message
├─ tool_calls[0]  id-Oslo   get_weather(59.9139, 10.7522)
└─ tool_calls[1]  id-Paris  get_weather(48.8566, 2.3522)
```

The real ids are long strings, so `id-Oslo` and `id-Paris` stand in for them below:
```
call_n3GF1jP3sf4D0Gv1C93rs7Je  ->  id-Oslo
call_ONVuf1IdcMcuWwYy4wp2OCIK  ->  id-Paris
```

The loop runs twice, once per entry of `tool_calls`. In **both** versions `get_weather` runs exactly twice and both cities are fetched correctly. The only difference is what ends up in `messages`.

##### Version A — the course code, append INSIDE the loop
```python
for tool_call in completion.choices[0].message.tool_calls:
    messages.append(completion.choices[0].message)   # runs every pass
    ...
    messages.append({"role": "tool", ...})
```

`messages` ends up with 6 entries:
```
1  system
2  user
3  assistant  asks for BOTH  id-Oslo + id-Paris   <- appended pass 1
4  tool       answers id-Oslo
5  assistant  asks for BOTH  id-Oslo + id-Paris   <- appended pass 2
6  tool       answers id-Paris
```

Entries 3 and 5 are not two similar messages. They are the **same object** in memory, appended twice:
```python
print(messages[2] is messages[4])
```
```
True
```

And this is the actual list, serialized the way the SDK sends it. The tool results are long, so their `content` is trimmed here; everything else is verbatim:

<div style="font-size: 0.85em">

```json
[
  {"role": "system",
   "content": "You are a helpful weather assistant."},

  {"role": "user",
   "content": "What's the weather like in Oslo and in Paris today?"},

  {"role": "assistant",
   "tool_calls": [
     {"id": "call_n3GF1jP3sf4D0Gv1C93rs7Je",
      "function": {"arguments": "{\"latitude\": 59.9139, \"longitude\": 10.7522}",
                   "name": "get_weather"},
      "type": "function"},
     {"id": "call_ONVuf1IdcMcuWwYy4wp2OCIK",
      "function": {"arguments": "{\"latitude\": 48.8566, \"longitude\": 2.3522}",
                   "name": "get_weather"},
      "type": "function"}
   ]},

  {"role": "tool",
   "tool_call_id": "call_n3GF1jP3sf4D0Gv1C93rs7Je",
   "content": "{\"current\": {\"time\": \"2026-09-20T18:30\", ...trimmed"},

  {"role": "assistant",
   "tool_calls": [
     {"id": "call_n3GF1jP3sf4D0Gv1C93rs7Je",
      "function": {"arguments": "{\"latitude\": 59.9139, \"longitude\": 10.7522}",
                   "name": "get_weather"},
      "type": "function"},
     {"id": "call_ONVuf1IdcMcuWwYy4wp2OCIK",
      "function": {"arguments": "{\"latitude\": 48.8566, \"longitude\": 2.3522}",
                   "name": "get_weather"},
      "type": "function"}
   ]},

  {"role": "tool",
   "tool_call_id": "call_ONVuf1IdcMcuWwYy4wp2OCIK",
   "content": "{\"current\": {\"time\": \"2026-09-20T18:30\", ...trimmed"}
]
```

</div>

The thing to see in that dump: **both ids sit inside a single assistant entry**, and that whole two-id entry appears twice.

##### Why the API refuses Version A
**The rule:** An assistant entry carrying `tool_calls` *opens a block*. The block is satisfied only when a `"role": "tool"` entry has appeared for **every** id in its `tool_calls` list. While any id is still outstanding, nothing but those tool entries may appear.

Walking Version A's six entries against that rule, tracking which ids are still owed:
```
entry 3  opens a block, needs: [id-Oslo, id-Paris]
entry 4  answers id-Oslo        still needed: [id-Paris]
entry 5  !! an assistant entry arrives while [id-Paris] is owed
entry 5  opens a block, needs: [id-Oslo, id-Paris]
entry 6  answers id-Paris       still needed: [id-Oslo]
end      !! list ends with [id-Oslo] still owed
```

Entry 3 asked for two results, but only Oslo arrived before entry 5 interrupted, so `id-Paris` was left hanging. The API says exactly that, and points at the entry that broke it:

<div style="font-size: 0.85em">

```
BadRequestError: Error code: 400 - {'error': {'message':
"An assistant message with 'tool_calls' must be followed by tool
messages responding to each 'tool_call_id'. The following
tool_call_ids did not have response messages:
call_ONVuf1IdcMcuWwYy4wp2OCIK",
'type': 'invalid_request_error',
'param': 'messages.[4].role', 'code': None}}
```

</div>

Two things worth reading out of that error:
- The unanswered id `call_ONVuf1...` is **id-Paris**, the second of entry 3's two requests.
- `'param': 'messages.[4].role'` is 0-indexed, so index 4 is **entry 5** — the duplicate assistant entry. The API is naming the message that broke the rule, not the missing answer.

Oslo is **not** fetched twice. `get_weather` still runs once per city; what gets duplicated is the model's *request* entry.

##### Version B — this notebook, append BEFORE the loop
```python
messages.append(completion.choices[0].message)   # runs once
for tool_call in completion.choices[0].message.tool_calls:
    ...
    messages.append({"role": "tool", ...})
```

`messages` ends up with 5 entries:
```
1  system
2  user
3  assistant  asks for BOTH  id-Oslo + id-Paris
4  tool       answers id-Oslo
5  tool       answers id-Paris
```

The same list serialized, trimmed the same way:

<div style="font-size: 0.85em">

```json
[
  {"role": "system",
   "content": "You are a helpful weather assistant."},

  {"role": "user",
   "content": "What's the weather like in Oslo and in Paris today?"},

  {"role": "assistant",
   "tool_calls": [
     {"id": "call_n3GF1jP3sf4D0Gv1C93rs7Je",
      "function": {"arguments": "{\"latitude\": 59.9139, \"longitude\": 10.7522}",
                   "name": "get_weather"},
      "type": "function"},
     {"id": "call_ONVuf1IdcMcuWwYy4wp2OCIK",
      "function": {"arguments": "{\"latitude\": 48.8566, \"longitude\": 2.3522}",
                   "name": "get_weather"},
      "type": "function"}
   ]},

  {"role": "tool",
   "tool_call_id": "call_n3GF1jP3sf4D0Gv1C93rs7Je",
   "content": "{\"current\": {\"time\": \"2026-09-20T18:30\", ...trimmed"},

  {"role": "tool",
   "tool_call_id": "call_ONVuf1IdcMcuWwYy4wp2OCIK",
   "content": "{\"current\": {\"time\": \"2026-09-20T18:30\", ...trimmed"}
]
```

</div>

Walking the same rule over it:
```
entry 3  opens a block, needs: [id-Oslo, id-Paris]
entry 4  answers id-Oslo        still needed: [id-Paris]
entry 5  answers id-Paris       still needed: []
end      block satisfied
```

One request, then one answer per id directly behind it, each tagged with the id it belongs to. Call 2 was accepted, `finish_reason` was `'stop'`, and the model wrote:

> Here's the current weather for today in both cities (local times):
> - Oslo: 18:30 local time, 17.2°C, wind 5.8 m/s
> - Paris: 18:30 local time, 23.3°C, wind 3.0 m/s

##### Why the course code still works in this notebook
With a **single** tool call the two versions produce **identical** histories: the loop runs once, so "once per pass" and "once, before the loop" are the same thing. That is why the course code works for the Oslo-only question here, and why the Section 5 dump shows a clean 4-message history.

The bug only surfaces when the model asks for two or more tools in one reply — and parallel tool calls are on by default, so any question naming two cities can trigger it.

In [10]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)


messages.append(completion.choices[0].message)  # the model's request: once, not once per tool call

for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [11]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_RxBcijsSCcyqeEV4DECORc9Z', function=Function(arguments='{"latitude":59.9139,"longitude":10.7522}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_RxBcijsSCcyqeEV4DECORc9Z',
  'content': '{"current": {"time": "2026-09-20T21:15", "interval": 900, "temperature_2m": 14.7, "wind_speed_10m": 3.2}, "units": {"time": "iso8601", "interval": "seconds", "temperature_2m": "\\u00b0C", "wind_speed_10m": "m/s"}, "timezone": "Europe/Oslo"}'}]

#### 5. The four messages the second call receives

**Key point:** `messages` is the model's entire memory. Call 2 reads all four entries, and entry 4 is the only place the weather data exists.

1. `system`: written by you.
2. `user`: written by you.
3. `assistant`: written by the model in call 1 and appended by you. It's a `ChatCompletionMessage` object, not a dict; the SDK accepts that and converts it to JSON when sending.
4. `tool`: written by you. It holds `get_weather`'s result as a string, and its `tool_call_id` matches the id in entry 3.

This is the `messages` part of call 2's request body, captured the same way as in Section 1:

<div style="font-size: 0.85em">

```json
[
  {"role": "system", "content": "You are a helpful weather assistant."},
  {"role": "user", "content": "What's the weather like in Oslo today?"},
  {
    "content": null,
    "role": "assistant",
    "tool_calls": [
      {
        "id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
        "function": {
          "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
          "name": "get_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
    "content": "{\"time\": \"2026-09-19T07:15\", \"interval\": 900, \"temperature_2m\": 14.5, \"wind_speed_10m\": 14.4}"
  }
]
```

</div>

The `\"` are escaped quotes: the arguments and the tool result are strings that contain JSON, sitting inside the outer JSON.

##### Why entry 3 shows some empty fields but not others
Entry 3 is a `ChatCompletionMessage`, and that class has seven fields: `content`, `refusal`, `role`, `annotations`, `audio`, `function_call`, `tool_calls`. The dump above carries only some of them — and the rule is *not* "drop the empty ones", because `"content": null` is empty and it is sent.

**What actually decides it:** Pydantic records which fields were present in the JSON the API sent back, in `model_fields_set`. When the SDK serializes the object into your next request, it sends exactly that set and omits fields that never appeared in the reply. An empty-but-present field like `"content": null` is in the set; a field the API never mentioned is not.

Checked against a live call-1 reply:
```python
reply = completion.choices[0].message
print(sorted(reply.model_fields_set))
```
```
['annotations', 'content', 'refusal', 'role', 'tool_calls']
```

Capturing the outgoing call-2 body (the same fake-transport trick as Section 1) shows entry 3 carrying precisely those five keys:
```json
{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "tool_calls": [ ... ]
}
```

`audio` and `function_call` exist on the class and are `None` on the object, but the API never returned them, so the SDK leaves them out.

**What this means for you:** Nothing to manage. Which empty fields turn up varies with what the API returned that day — the saved dump above has no `"refusal"`, the live capture has `"refusal": null` — and none of them carry information. The only part of entry 3 that call 2 actually needs is `tool_calls`, because that is what entry 4's `tool_call_id` points at.

In [12]:
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )


#### 6. Call 2: the model writes the answer, and `Field()` descriptions get read

**Key point:** Call 2 sends the four messages, the same `tools`, and the `WeatherResponse` schema. The model now has the weather data (entry 4), so it writes the final answer instead of asking for a tool.

This is the `response_format` part of call 2's request body. It's how your `Field(description=...)` text reaches the model:

<div style="font-size: 0.85em">

```json
"response_format": {
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": {
        "temperature": {
          "description": "The current temperature in celsius for the given location.",
          "title": "Temperature",
          "type": "number"
        },
        "response": {
          "description": "A natural language response to the user's question.",
          "title": "Response",
          "type": "string"
        }
      },
      "required": ["temperature", "response"],
      "title": "WeatherResponse",
      "type": "object",
      "additionalProperties": false
    },
    "name": "WeatherResponse",
    "strict": true
  }
}
```

</div>

##### Where that JSON comes from, and how to print it yourself
**Key point:** You never write `response_format` by hand. You pass the *class* `WeatherResponse`, and the SDK converts it into that JSON Schema before sending. The model never sees your Python class, never sees `Field`, and never sees Pydantic. It only ever sees the JSON above.

**How to print it yourself.** The SDK's converter is an ordinary function you can import and call. It makes no network request:
```python
from openai.lib._parsing._completions import type_to_response_format_param

print(json.dumps(type_to_response_format_param(WeatherResponse), indent=2))
```
That prints exactly the block shown above. The leading underscores mark it an SDK internal, so it is fine for inspecting but not something to build on.

**The conversion, in three steps.** `parse()` puts your class through this chain:

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
WeatherResponse  <span style="opacity: 0.55">(your Pydantic class)</span>
   │  <span style="opacity: 0.7">parse(), in resources/chat/completions/completions.py</span>
   ▼
type_to_response_format_param()   <span style="opacity: 0.55">lib/_parsing/_completions.py</span>
   │    name = response_format.__name__   <span style="opacity: 0.7">-> "WeatherResponse"</span>
   │    "strict": True                    <span style="opacity: 0.7">-> hardcoded by the SDK</span>
   ▼
to_strict_json_schema()           <span style="opacity: 0.55">lib/_pydantic.py</span>
   │    model_json_schema(model)          <span style="opacity: 0.7">-> plain Pydantic</span>
   │    _ensure_strict_json_schema(...)   <span style="opacity: 0.7">-> additionalProperties</span>
   ▼
the JSON above
</pre>

Trimmed to the lines that build the dict, that is the real SDK source:
```python
# openai/lib/_parsing/_completions.py
def type_to_response_format_param(response_format):
    ...
    name = response_format.__name__
    return {
        "type": "json_schema",
        "json_schema": {
            "schema": to_strict_json_schema(json_schema_type),
            "name": name,
            "strict": True,
        },
    }

# openai/lib/_pydantic.py
def to_strict_json_schema(model):
    schema = model_json_schema(model)
    return _ensure_strict_json_schema(schema, path=(), root=schema)

# openai/lib/_pydantic.py, inside _ensure_strict_json_schema
if typ == "object" and "additionalProperties" not in json_schema:
    json_schema["additionalProperties"] = False
```

**Step 1 is pure Pydantic, with no OpenAI involved.** `model_json_schema()` is a Pydantic method and works on any `BaseModel`:
```python
print(json.dumps(WeatherResponse.model_json_schema(), indent=2))
```
```json
{
  "properties": {
    "temperature": {
      "description": "The current temperature in celsius for the given location.",
      "title": "Temperature",
      "type": "number"
    },
    "response": {
      "description": "A natural language response to the user's question.",
      "title": "Response",
      "type": "string"
    }
  },
  "required": ["temperature", "response"],
  "title": "WeatherResponse",
  "type": "object"
}
```

Set that beside the finished block above: the only thing the OpenAI SDK adds to the schema itself is `"additionalProperties": false`. Everything else it adds is the `"type"` / `"name"` / `"strict"` wrapper around it.

**So does the model see `Field()`?** No. It sees the strings you put *inside* `Field`. The chain is `Field(description=...)` → Pydantic writes it into the schema as `"description"` → the SDK forwards that schema → the model reads it. The word `Field`, and Python itself, never leave your machine.

##### Which line of the class produced which part of the JSON
- `class WeatherResponse` → `"title": "WeatherResponse"` inside the schema, and `"name": "WeatherResponse"` on the wrapper (that one is `response_format.__name__`).
- `temperature: float` → `"type": "number"`. Pydantic maps `float` to `number`, `str` to `string`, `int` to `integer`, `bool` to `boolean`, and `list[...]` to `array`.
- The field name `temperature` → the property key, plus `"title": "Temperature"`, which Pydantic generates by title-casing the field name. Nothing of yours sets it, and the model gains nothing from it.
- `Field(description="The current temperature...")` → `"description": "..."`. **This is the only part of the schema you wrote in prose.** It is the counterpart of the tool's `description` from Section 1: the place where you tell the model what a field means.
- **Neither field has a default value** → both names appear in `"required": ["temperature", "response"]`. Give one a default (`response: str = "n/a"`) and it drops out of `required`.
- `"additionalProperties": false` → added by the SDK, not by you. It forbids the model inventing extra keys.
- `"strict": true` → hardcoded by the SDK. It makes the schema a hard constraint on generation rather than a suggestion, which is why `.parsed` can be trusted to fit your class.

Drop the `Field(...)` wrappers and the descriptions simply vanish; everything else is identical:
```python
class NoDescriptions(BaseModel):
    temperature: float
    response: str
```
```json
"properties": {
  "temperature": {"title": "Temperature", "type": "number"},
  "response": {"title": "Response", "type": "string"}
}
```
The model would still be forced to return those two keys with those two types, but it would have to guess what they *mean* — the same failure mode Section 7 documents for the missing units.

**Why `tools=tools` is passed again:** The API stores nothing between calls, so each request has to describe everything again. If you want the model to still know `get_weather` exists (and be able to call it again), its definition has to be in this request too.

**What the code assumes:** Call 2 could come back with *another* tool request instead of an answer. Then `finish_reason` would be `'tool_calls'` and `.parsed` would be `None`. This course code assumes one round trip is enough. Real agents repeat steps 2 and 3 of the flow in a loop until `finish_reason` is `'stop'`.

In [13]:
completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=WeatherResponse,
)

##### Seeing `completion_2` itself
The cell above only *assigns* `completion_2`, so Jupyter prints nothing and the object stays invisible. The next cell prints it. Three things are worth looking at:

- **`finish_reason` is now `'stop'`**, not `'tool_calls'`. The model wrote an answer instead of asking for another tool, so `message.tool_calls` is `None`.
- **`message.content` is a `str`** holding the JSON the model generated, shaped by the schema above.
- **`message.parsed` is a `WeatherResponse` object**, and only `parse()` adds it. It is `content` fed through your class, so `parsed.temperature` is a real `float` you can do arithmetic on.

`content` and `parsed` are the same data in two forms, text and object — the pair from Exhaustive 2.

Run on a fresh call 2 with this notebook's exact four messages, it printed:

<div style="font-size: 0.85em">

```
finish_reason: stop
tool_calls:    None

message.content   str
'{"temperature":14.5,"response":"In Oslo right now (as of 07:15
today), it's about 14.5°C with winds around 14 km/h. ..."}'

message.parsed    WeatherResponse
WeatherResponse(temperature=14.5, response='In Oslo right now (as
of 07:15 today), it's about 14.5°C with winds around 14 km/h. ...')

same data? True
```

</div>

The wording differs from the saved `final_response` output below because that came from the original run, and the model writes fresh prose every time. Worth noticing: this re-run said "14 km/h", which is the *correct* unit, while the saved run said "14 m/s", which is wrong. Section 7 explains why that difference is luck rather than knowledge — nothing in the request states the wind unit either way, so the model is guessing both times.

In [23]:
print("\nType of completion_2: \n", type(completion_2))
print("\n")
pprint(completion_2.model_dump(warnings=False), sort_dicts=False)


Type of completion_2: 
 <class 'openai.types.chat.parsed_chat_completion.ParsedChatCompletion[TypeVar]'>


{'id': 'chatcmpl-EQHTAI0bsYdLK5jugwdO51e3SX7N0',
 'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'content': '{"temperature":14.7,"response":"Right '
                                     "now in Oslo it's 14.7°C with a light "
                                     'wind around 3.2 m/s. Local time is 21:15 '
                                     '(Europe/Oslo)."}',
                          'refusal': None,
                          'role': 'assistant',
                          'annotations': [],
                          'audio': None,
                          'function_call': None,
                          'tool_calls': None,
                          'parsed': {'temperature': 14.7,
                                     'response': "Right now in Oslo it's "
                                                 

In [15]:
print("finish_reason:", completion_2.choices[0].finish_reason)
print("tool_calls:   ", completion_2.choices[0].message.tool_calls)
print()

content = completion_2.choices[0].message.content
parsed = completion_2.choices[0].message.parsed

print("message.content  ", type(content).__name__)
print(repr(content))
print()
print("message.parsed   ", type(parsed).__name__)
print(repr(parsed))
print()
print("same data?", json.loads(content) == parsed.model_dump())

finish_reason: stop
tool_calls:    None

message.content   str
'{"temperature":14.7,"response":"Right now in Oslo it\'s 14.7°C with a light wind around 3.2 m/s. Local time is 21:15 (Europe/Oslo)."}'

message.parsed    WeatherResponse
WeatherResponse(temperature=14.7, response="Right now in Oslo it's 14.7°C with a light wind around 3.2 m/s. Local time is 21:15 (Europe/Oslo).")

same data? True


In [16]:
final_response = completion_2.choices[0].message.parsed
print(final_response.temperature)
print(final_response.response)

14.7
Right now in Oslo it's 14.7°C with a light wind around 3.2 m/s. Local time is 21:15 (Europe/Oslo).


#### 7. Check the answer against what the model was given

**Key point:** The model only knows what is in `messages`. In the saved run the tool message carried bare numbers with no units and no timezone, so the model guessed both, and got both wrong.

##### First: where the four fields in the tool message come from
Section 5 shows the tool message holding this string:
```
{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}
```
Nothing added those four fields to the result. They **are** one block of Open-Meteo's reply, `data["current"]`, and the run that produced this notebook used the course's return line, `return data["current"]`.

Open-Meteo's full reply is much larger. A live request for Oslo (2026-09-20), with the long `hourly` arrays cut short:
```json
{
  "latitude": 59.9, "longitude": 10.75, "elevation": 37.0,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "current_units": {"time": "iso8601", "interval": "seconds",
                    "temperature_2m": "°C", "wind_speed_10m": "km/h"},
  "current":       {"time": "2026-09-20T07:45", "interval": 900,
                    "temperature_2m": 13.3, "wind_speed_10m": 19.1},
  "hourly_units":  {"time": "iso8601", "temperature_2m": "°C", ...},
  "hourly":        {"time": ["2026-09-20T00:00", ...], "temperature_2m": [...], ...}
}
```
`data["current"]` is the block labelled `"current"`: those four keys, nothing else. `json.dumps` turned that one dict into the tool message string, which is why the fields sit flat at the top level. `interval: 900` is Open-Meteo's own bookkeeping (the reading refreshes every 900 seconds); it rides along inside the block.

What the course's return line dropped: `current_units`, `timezone`, `hourly_units` and `hourly`.

##### Why the `get_weather` cell above doesn't match the output below it
The `get_weather` cell near the top of this notebook is the **fixed** version. The outputs saved in this notebook come from a run made **before** that fix, when the function was still the course's one-liner. The two shapes:

```python
# course code — produced every saved output in this notebook
return data["current"]
# tool message content:
# '{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}'

# fixed code — the cell at the top of this notebook
return {"current": data["current"], "units": data["current_units"], "timezone": data["timezone"]}
# tool message content (live values for Oslo, 2026-09-20):
# '{"current": {"time": "2026-09-20T09:45", "interval": 900, "temperature_2m": 13.3,
#   "wind_speed_10m": 5.3}, "units": {"time": "iso8601", "interval": "seconds",
#   "temperature_2m": "°C", "wind_speed_10m": "m/s"}, "timezone": "Europe/Oslo"}'
```

The numbers stay where they were, one level deeper, under `"current"`. Re-running this notebook replaces the flat tool message with the nested one; `Supplement_3-tools.ipynb` already holds such a run.

##### What the model got wrong, and why
- **Wind: right number, wrong label.** `"wind_speed_10m": 14.4` arrived with no unit. The course URL requests no unit, and Open-Meteo's default is km/h — the reply says so in `current_units`, which the return line dropped. The model wrote "around 14 m/s". Nothing was miscalculated; the number was simply called by the wrong name, and 14.4 km/h is 4 m/s, so the answer overstates the wind 3.6-fold.
- **Time: right instant, wrong clock.** `"time": "2026-09-19T07:15"` arrived with no timezone. The reply's `timezone` was `"GMT"`, also dropped. Oslo in September runs on UTC+2, so that instant is 09:15 there. The model wrote "07:15 local time".
- **Temperature: right, for an instructive reason.** `temperature_2m` is in °C, and the word "celsius" appears twice in the request: in the tool's `description` and in the `Field` description. That unit reached the model through the **tool definition**, not through the data. The two values with no label anywhere in the request are exactly the two it got wrong.

##### The fix: two halves, both needed
The same URL twice at the same moment, one plain and one with the two extra parameters (live for Oslo, 2026-09-20):

| | course URL | `&timezone=auto&wind_speed_unit=ms` |
| --- | --- | --- |
| `current.time` | `"2026-09-20T07:45"` | `"2026-09-20T09:45"` |
| `current.wind_speed_10m` | `19.1` | `5.3` |
| `current_units.wind_speed_10m` | `"km/h"` | `"m/s"` |
| `timezone` | `"GMT"` | `"Europe/Oslo"` |
| `current.temperature_2m` | `13.3` | `13.3` |

Both columns describe the same weather: 19.1 km/h **is** 5.3 m/s, and 07:45 GMT **is** 09:45 in Oslo.

1. **The URL parameters change which numbers arrive.** On their own they fix nothing for the model, because `current` still carries no labels. The model would still be guessing; it would just happen to guess right.
2. **Returning `current_units` and `timezone` puts the labels into the tool message.** That is what removes the guess.

One half makes the values what you want, the other tells the model what they are. Same lesson as Section 1: if the model should know something, it has to be in the request.